In [1]:
!pip install -q openai-agents python-dotenv


[notice] A new release of pip available: 22.3.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

In [3]:
from agents import Agent, Runner, tool, function_tool
from agents.handoffs import Handoff
from agents.tracing import trace

In [4]:
from agents import Agent

math_agent = Agent(
    name="Math Agent",
    instructions="""
    You are a mathematical problem solver.
    Solve problems step by step and give final answer clearly.
    """,
    model="gpt-4.1-mini"
)

finance_agent = Agent(
    name="Finance Agent",
    instructions="""
    You are a financial analyst.
    Provide structured financial reasoning.
    """,
    model="gpt-4.1-mini"
)

In [5]:
!pip install -q openai-agents


[notice] A new release of pip available: 22.3.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from agents import Agent, Runner
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API Key: ")

In [7]:
math_agent = Agent(
    name="Math Agent",
    instructions="""
    You are a mathematical reasoning expert.
    Solve problems step by step and give final answer clearly.
    """,
    model="gpt-4.1-mini"
)

finance_agent = Agent(
    name="Finance Agent",
    instructions="""
    You are a financial advisor.
    Provide structured reasoning and safe financial advice.
    """,
    model="gpt-4.1-mini"
)

In [8]:
orchestrator = Agent(
    name="Orchestrator",
    instructions="""
    If the query is about math → handoff to Math Agent.
    If about finance → handoff to Finance Agent.
    Otherwise answer yourself.
    """,
    handoffs=[math_agent, finance_agent],   # ← FIXED
    model="gpt-4.1"
)

In [12]:
import asyncio
from agents import Agent, Runner

math_agent = Agent(
    name="Math Agent",
    instructions="Solve math problems step by step.",
    model="gpt-4.1-mini"
)

finance_agent = Agent(
    name="Finance Agent",
    instructions="Answer finance questions responsibly.",
    model="gpt-4.1-mini"
)

orchestrator = Agent(
    name="Orchestrator",
    instructions="""
    Route math queries to Math Agent.
    Route finance queries to Finance Agent.
    """,
    handoffs=[math_agent, finance_agent],
    model="gpt-4.1"
)

async def main():
    result = await Runner.run(
        orchestrator,
        "What is 56 * 89?"
    )
    print(result.final_output)

await main()

Tool name 'transfer_to_Math Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_math_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Finance Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_finance_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Math Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_math_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Finance Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_finance_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


Let's calculate \( 56 \times 89 \) step by step:

1. Multiply 56 by 80:
   \[
   56 \times 80 = 4480
   \]

2. Multiply 56 by 9:
   \[
   56 \times 9 = 504
   \]

3. Add the two results:
   \[
   4480 + 504 = 4984
   \]

So, \( 56 \times 89 = 4984 \).


### **Static Handoff (Predefined Delegation)**

In [13]:
math_agent = Agent(
    name="Math Specialist",
    instructions="Solve mathematical problems step by step.",
    model="gpt-4.1-mini"
)

legal_agent = Agent(
    name="Legal Specialist",
    instructions="Provide structured legal reasoning.",
    model="gpt-4.1-mini"
)

orchestrator = Agent(
    name="Orchestrator",
    instructions="""
    If query involves calculations → use Math Specialist.
    If legal matter → use Legal Specialist.
    Otherwise answer directly.
    """,
    handoffs=[math_agent, legal_agent],
    model="gpt-4.1"
)

async def run_static():
    result = await Runner.run(
        orchestrator,
        "What is 45 * 78?"
    )
    print(result.final_output)

await run_static()

Tool name 'transfer_to_Math Specialist' contains invalid characters for function calling and has been transformed to 'transfer_to_math_specialist'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Legal Specialist' contains invalid characters for function calling and has been transformed to 'transfer_to_legal_specialist'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Math Specialist' contains invalid characters for function calling and has been transformed to 'transfer_to_math_specialist'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Legal Specialist' contains invalid characters for function calling and has been transformed to 'transfer_to_legal_specialist'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


Let's calculate \( 45 \times 78 \) step by step:

1. Break down \( 78 \) into \( 70 + 8 \).
2. Multiply \( 45 \times 70 = 3150 \).
3. Multiply \( 45 \times 8 = 360 \).
4. Add the two results: \( 3150 + 360 = 3510 \).

So, \( 45 \times 78 = 3510 \).


### **Conditional Programmatic Handoff**

In [14]:
async def conditional_router(user_query: str):
    if "calculate" in user_query.lower():
        return await Runner.run(math_agent, user_query)
    elif "law" in user_query.lower():
        return await Runner.run(legal_agent, user_query)
    else:
        return await Runner.run(orchestrator, user_query)

result = await conditional_router("Calculate 100 * 45")
print(result.final_output)

To calculate \(100 \times 45\):

Step 1: Multiply the numbers:
\[100 \times 45 = 4500\]

Answer: \(4500\)


### **Tool-Based Handoff (Agent-as-Tool)**

In [15]:
@function_tool
async def math_tool(query: str) -> str:
    result = await Runner.run(math_agent, query)
    return result.final_output

main_agent = Agent(
    name="Main Agent",
    instructions="""
    If math-related question appears, call math_tool.
    """,
    tools=[math_tool],
    model="gpt-4.1"
)

result = await Runner.run(main_agent, "What is 234 * 567?")
print(result.final_output)

234 multiplied by 567 is 132,678.


### **TYPE 4 — Multi-Hop Sequential Handoff**

In [16]:
analysis_agent = Agent(
    name="Analysis Agent",
    instructions="Analyze problem deeply.",
    model="gpt-4.1-mini"
)

execution_agent = Agent(
    name="Execution Agent",
    instructions="Perform required calculation or solution.",
    model="gpt-4.1-mini"
)

analysis_agent.handoffs = [execution_agent]

async def multi_hop():
    result = await Runner.run(
        analysis_agent,
        "Break down and solve 345 * 987"
    )
    print(result.final_output)

await multi_hop()

Tool name 'transfer_to_Execution Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_execution_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Execution Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_execution_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


To break down and solve \( 345 \times 987 \), we can use the distributive property of multiplication. Here's the step-by-step breakdown:

1. Express 987 as \( 900 + 80 + 7 \).
2. Multiply 345 by each of these parts.
3. Add the results together.

Step 1: \( 345 \times 900 \)  
Step 2: \( 345 \times 80 \)  
Step 3: \( 345 \times 7 \)  

Now let's compute each:

- \( 345 \times 900 = 345 \times (9 \times 100) = (345 \times 9) \times 100 = 3105 \times 100 = 310,500 \)  
- \( 345 \times 80 = 345 \times (8 \times 10) = (345 \times 8) \times 10 = 2760 \times 10 = 27,600 \)  
- \( 345 \times 7 = 2415 \)  

Finally, add the three results:  
\( 310,500 + 27,600 + 2,415 = 340,515 \)

So, \( 345 \times 987 = 340,515 \).


### **TYPE 5 — Router Agent (Dynamic LLM Routing)**

In [17]:
router = Agent(
    name="Smart Router",
    instructions="""
    Carefully analyze intent.
    Route to appropriate agent.
    """,
    handoffs=[math_agent, legal_agent],
    model="gpt-4.1"
)

result = await Runner.run(router, "Explain contract breach law.")
print(result.final_output)

Tool name 'transfer_to_Math Specialist' contains invalid characters for function calling and has been transformed to 'transfer_to_math_specialist'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Legal Specialist' contains invalid characters for function calling and has been transformed to 'transfer_to_legal_specialist'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Math Specialist' contains invalid characters for function calling and has been transformed to 'transfer_to_math_specialist'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Legal Specialist' contains invalid characters for function calling and has been transformed to 'transfer_to_legal_specialist'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


**Contract breach law** refers to the legal rules and principles governing what happens when one party fails to fulfill their obligations under a contract. Here’s an overview:

### 1. **What is a Contract Breach?**
A breach of contract occurs when one party breaks, fails to perform, or refuses to fulfill the promises made in a binding agreement without lawful excuse.

### 2. **Types of Breaches**
- **Minor (Partial) Breach:** A small or less critical failure that doesn’t destroy the value of the contract as a whole.
- **Material Breach:** A major failure that defeats the contract’s purpose, allowing the other party to terminate the contract and seek damages.
- **Anticipatory Breach:** One party clearly indicates, before their performance is due, that they will not fulfill the contract.

### 3. **Elements of a Breach of Contract Claim**
- **Valid Contract:** There must be a legally binding agreement.
- **Breach:** Proof that one party failed to perform as required by the contract.
- **P

### **TYPE 6 — Parallel Multi-Agent Execution**

In [18]:
async def parallel_agents():
    task1 = Runner.run(math_agent, "Solve 123 * 456")
    task2 = Runner.run(legal_agent, "Define tort law")

    results = await asyncio.gather(task1, task2)

    for r in results:
        print("\n--- Output ---")
        print(r.final_output)

await parallel_agents()


--- Output ---
Let's solve \(123 \times 456\) step by step:

1. Multiply 123 by 6 (the units digit of 456):
   \[
   123 \times 6 = 738
   \]

2. Multiply 123 by 5 (the tens digit of 456), then multiply by 10:
   \[
   123 \times 5 = 615 \implies 615 \times 10 = 6150
   \]

3. Multiply 123 by 4 (the hundreds digit of 456), then multiply by 100:
   \[
   123 \times 4 = 492 \implies 492 \times 100 = 49200
   \]

4. Add all the results together:
   \[
   738 + 6150 + 49200 = 56088
   \]

So, \(123 \times 456 = \boxed{56088}\).

--- Output ---
**Definition of Tort Law**

Tort law is a branch of civil law that deals with situations where one party's wrongful conduct causes harm or loss to another, thereby giving rise to a legal claim for compensation. Unlike criminal law, which punishes wrongs against the state, tort law is primarily concerned with providing remedies to individuals who have suffered injuries or damages.

---

**Structured Legal Reasoning**

1. **Purpose of Tort Law**  
   

In [22]:
import asyncio
from typing import List

from pydantic import BaseModel
from agents import (
    Agent,
    GuardrailFunctionOutput,
    RunContextWrapper,
    Runner,
    TResponseInputItem,
    input_guardrail,
    Guardrail,
)

# -----------------------------
# 1️⃣ Output Schema
# -----------------------------
class ChurnDetectionOutput(BaseModel):
    is_churn_risk: bool
    reasoning: str


# -----------------------------
# 2️⃣ Churn Detection Agent
# -----------------------------
churn_detection_agent = Agent(
    name="Churn Detection Agent",
    instructions=(
        "Identify if the user message indicates a potential "
        "customer churn risk. "
        "Return is_churn_risk as True or False and explain reasoning."
    ),
    output_type=ChurnDetectionOutput,
)


# -----------------------------
# 3️⃣ Guardrail Function
# -----------------------------
@input_guardrail
async def churn_detection_tripwire(
    ctx: RunContextWrapper,
    agent: Agent,
    input: List[TResponseInputItem],
) -> GuardrailFunctionOutput:

    result = await Runner.run(
        churn_detection_agent,
        input,
        context=ctx.context,
    )

    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=result.final_output.is_churn_risk,
    )


# -----------------------------
# 4️⃣ Customer Support Agent
# -----------------------------
customer_support_agent = Agent(
    name="Customer Support Agent",
    instructions=(
        "You are a helpful customer support agent. "
        "Answer user questions politely and clearly."
    ),
    input_guardrails=[
        Guardrail(guardrail_function=churn_detection_tripwire),
    ],
)


# -----------------------------
# 5️⃣ Main Execution
# -----------------------------
async def main():

    print("\n---- SAFE MESSAGE ----")
    try:
        result = await Runner.run(
            customer_support_agent,
            "Hello! I have a question about my billing."
        )
        print("Response:", result.final_output)

    except Exception as e:
        print("⚠ Guardrail triggered unexpectedly!")
        print(e)

    print("\n---- CHURN MESSAGE ----")
    try:
        result = await Runner.run(
            customer_support_agent,
            "I am very unhappy with your service and I want to cancel my subscription."
        )
        print("Response:", result.final_output)

    except Exception as e:
        print("⚠ Guardrail Triggered!")
        print("Reason:", e)


if __name__ == "__main__":
    asyncio.run(main())

ImportError: cannot import name 'Guardrail' from 'agents' (c:\Program Files\Python311\Lib\site-packages\agents\__init__.py)